# Smart Wallet: Construyendo Agentes con Tool Calling y Automatización
**CONTECS 2026** · Facultad de Ingeniería de Sistemas Computacionales (FISC - UTP)
Martes 20 de octubre de 2026 · 3:15 p. m. a 5:15 p. m. · Ponente: Isaías Reyes

**Instrucciones iniciales:**
1. Haz clic en **Archivo > Guardar una copia en Drive**.
2. Completa los bloques marcados con `# TODO` junto con el instructor.
3. Si Colab se reinicia o aparece `NameError`, haz clic en la celda donde va la clase y usa **Entorno de ejecución > Ejecutar anteriores** (`Ctrl + F8`).

---
# Parte 1 · Setup y Conexión
1. Crea tu clave en [aistudio.google.com/apikey](https://aistudio.google.com/apikey) con una cuenta personal de Gmail.
2. En el panel izquierdo de Colab (**Secrets / icono de llave**), agrega el secreto `GOOGLE_API_KEY` y activa **Notebook access**.

In [ ]:
%pip install -q google-genai pyTelegramBotAPI yfinance pydantic pandas

In [ ]:
import logging
from google.colab import userdata
from google import genai
from google.genai import types

logging.getLogger("google_genai.models").setLevel(logging.ERROR)

GOOGLE_API_KEY = userdata.get("GOOGLE_API_KEY")
client = genai.Client(api_key=GOOGLE_API_KEY)
MODELO = "gemini-3.5-flash"

saludo = client.models.generate_content(
    model=MODELO,
    contents="Saluda en una sola frase a los estudiantes de la FISC en CONTECS 2026.",
)
print("Conectado al modelo:", MODELO)
print("Gemini:", saludo.text)

---
# Parte 2 · Experimentos: Límites de un LLM sin herramientas

In [ ]:
# Experimento 1: Datos privados y precios en tiempo real
pregunta_1 = "¿Cuánto dinero tengo disponible en mi cuenta de Yappy y cuál es el precio exacto de Bitcoin en este momento?"

respuesta_1 = client.models.generate_content(model=MODELO, contents=pregunta_1)
print(respuesta_1.text)

In [ ]:
# Experimento 2: Matemática probabilística (LLM) vs. Cálculo determinista (Python)
pregunta_2 = (
    "Si invierto 1234.56 dólares al 7.3% anual con capitalización mensual y además aporto "
    "47.89 dólares al final de cada mes durante 17 años, ¿cuánto tendré al final? "
    "Responde SOLO con el número final, sin texto adicional."
)
respuesta_2 = client.models.generate_content(model=MODELO, contents=pregunta_2)
print("Gemini (predice texto):", respuesta_2.text.strip())

P = 1234.56
aporte = 47.89
tasa_anual = 0.073
anios = 17

i = tasa_anual / 12
n = anios * 12
valor_exacto = P * (1 + i) ** n + aporte * ((1 + i) ** n - 1) / i
print(f"Python (calcula):       {valor_exacto:,.2f}")

---
# Parte 3 · Ejercicio 1: Tu primera herramienta (Tool Calling)

In [ ]:
BILLETERA_DEMO = {
    "efectivo": 45.75,
    "banco": 812.30,
    "yappy": 23.10,
}


def consultar_saldo_cuenta(cuenta: str) -> dict:
    # TODO 1.1: Escribe aquí el docstring (entre triples comillas) que explique qué hace la función
    #           y cuáles son las opciones válidas para cuenta: 'efectivo', 'banco' o 'yappy'.

    cuenta = cuenta.strip().lower()
    print(f"   [TOOL EJECUTADA] consultar_saldo_cuenta(cuenta='{cuenta}')")

    if cuenta not in BILLETERA_DEMO:
        return {"ok": False, "error": f"La cuenta '{cuenta}' no existe. Usa: efectivo, banco o yappy."}

    # TODO 1.2: Devuelve un diccionario con "ok": True, "cuenta": cuenta y "saldo_usd": BILLETERA_DEMO[cuenta]


In [ ]:
# TODO 1.3: Conecta la función pasándola dentro de la lista tools=[...]
respuesta = client.models.generate_content(
    model=MODELO,
    contents="¿Cuánta plata tengo en Yappy y cuánto en efectivo? ¿Me alcanza lo de Yappy para una pizza de $18?",
    config=types.GenerateContentConfig(
        # Escribe aquí la línea tools=[...],
    ),
)

print("\nRespuesta del agente:")
print(respuesta.text)

In [ ]:
# Radiografía del ciclo interno de Tool Calling
historial = respuesta.automatic_function_calling_history or []

for numero, mensaje in enumerate(historial, start=1):
    for parte in mensaje.parts:
        if parte.text:
            print(f"{numero}. [{mensaje.role}] texto: {parte.text}")
        if parte.function_call:
            print(f"{numero}. [{mensaje.role}] function_call: {parte.function_call.name}({parte.function_call.args})")
        if parte.function_response:
            print(f"{numero}. [{mensaje.role}] function_response: {parte.function_response.response}")

print(f"{len(historial) + 1}. [model] respuesta final: {respuesta.text}")

---
# Parte 4 · Ejercicio 2: Memoria persistente con SQLite

In [ ]:
import sqlite3
import pandas as pd

conn = sqlite3.connect("smart_wallet.db", check_same_thread=False)
cursor = conn.cursor()

cursor.execute("""
CREATE TABLE IF NOT EXISTS movimientos (
    id          INTEGER PRIMARY KEY AUTOINCREMENT,
    tipo        TEXT NOT NULL,
    categoria   TEXT NOT NULL,
    monto       REAL NOT NULL,
    descripcion TEXT NOT NULL,
    fecha       TEXT DEFAULT (datetime('now', '-5 hours'))
)
""")
conn.commit()


def ver_tabla():
    return pd.read_sql_query("SELECT * FROM movimientos ORDER BY id", conn)


print("Base de datos 'smart_wallet.db' lista.")

In [ ]:
def registrar_movimiento(tipo: str, categoria: str, monto: float, descripcion: str) -> dict:
    """Guarda un ingreso o un gasto en la base de datos.
    Úsala SIEMPRE que el usuario diga que cobró, recibió, ganó, gastó, pagó o compró algo.

    Args:
        tipo: Exactamente 'ingreso' (entra dinero) o 'gasto' (sale dinero).
        categoria: Una de: 'comida', 'transporte', 'entretenimiento', 'educacion', 'salud', 'servicios', 'ingresos', 'otros'.
        monto: Cantidad en dólares, número positivo. Ejemplo: 12.50
        descripcion: Detalle breve. Ejemplos: 'almuerzo en la fonda', 'beca IFARHU'.
    """
    tipo = tipo.strip().lower()
    categoria = categoria.strip().lower()
    monto = round(float(monto), 2)

    if tipo not in ("ingreso", "gasto"):
        return {"ok": False, "error": "El tipo debe ser 'ingreso' o 'gasto'."}
    if monto <= 0:
        return {"ok": False, "error": "El monto debe ser mayor que cero."}

    # TODO 2.1: Guarda el movimiento en la tabla movimientos con cursor.execute(...).
    #           Usa placeholders ? para los 4 valores (nunca pegues las variables dentro del texto SQL)
    #           y después guarda los cambios con conn.commit()

    nuevo_id = cursor.lastrowid
    print(f"   [SQL INSERT] #{nuevo_id} {tipo} de ${monto:.2f} en '{categoria}' ({descripcion})")
    return {"ok": True, "id": nuevo_id, "tipo": tipo, "categoria": categoria, "monto_usd": monto}


def consultar_billetera() -> dict:
    """Consulta la base de datos y devuelve: ingresos totales, gastos totales, saldo disponible,
    gastos por categoría y los últimos 5 movimientos con su id.
    """
    print("   [SQL SELECT] Consultando saldo y resumen")

    cursor.execute("SELECT SUM(monto) FROM movimientos WHERE tipo = 'ingreso'")
    ingresos = cursor.fetchone()[0] or 0.0

    cursor.execute("SELECT SUM(monto) FROM movimientos WHERE tipo = 'gasto'")
    gastos = cursor.fetchone()[0] or 0.0

    cursor.execute(
        "SELECT categoria, SUM(monto) FROM movimientos WHERE tipo = 'gasto' "
        "GROUP BY categoria ORDER BY SUM(monto) DESC"
    )
    gastos_por_categoria = {}
    for categoria, total in cursor.fetchall():
        gastos_por_categoria[categoria] = round(total, 2)

    cursor.execute("SELECT id, tipo, categoria, monto, descripcion FROM movimientos ORDER BY id DESC LIMIT 5")
    ultimos_movimientos = []
    for fila in cursor.fetchall():
        ultimos_movimientos.append(
            {"id": fila[0], "tipo": fila[1], "categoria": fila[2], "monto": fila[3], "descripcion": fila[4]}
        )

    return {
        "ok": True,
        "ingresos_totales_usd": round(ingresos, 2),
        "gastos_totales_usd": round(gastos, 2),
        "saldo_disponible_usd": round(ingresos - gastos, 2),
        "gastos_por_categoria": gastos_por_categoria,
        "ultimos_movimientos": ultimos_movimientos,
    }


def eliminar_movimiento(id_movimiento: int) -> dict:
    """Elimina un movimiento de la base de datos usando su id numérico.

    Args:
        id_movimiento: Número de id del movimiento que se desea borrar.
    """
    print(f"   [SQL DELETE] Borrando el movimiento #{id_movimiento}")
    # TODO 2.2: Borra la fila cuyo id sea id_movimiento con cursor.execute(...) y un placeholder ?,
    #           y después guarda los cambios con conn.commit()

    if cursor.rowcount == 0:
        return {"ok": False, "error": f"No existe ningún movimiento con el id {id_movimiento}."}
    return {"ok": True, "id_eliminado": id_movimiento}

In [ ]:
INSTRUCCIONES_AGENTE = """
Eres Smart Wallet, un asistente de finanzas personales para estudiantes universitarios en Panamá.
Reglas:
1. Usa SIEMPRE tus herramientas para registrar movimientos y para consultar saldos o precios.
2. Si tienes una herramienta para un cálculo (simular_inversion o calcular_itbms_panama), úsala en lugar de calcular de memoria.
3. Nunca inventes datos.
4. Antes de borrar un movimiento, confirma con el usuario cuál es.
5. Responde en español, en texto plano (sin asteriscos ni tablas), de forma breve y clara.
6. Lo que digas sobre inversiones es información educativa, no asesoría financiera.
"""

chat_billetera = client.chats.create(
    model=MODELO,
    config=types.GenerateContentConfig(
        system_instruction=INSTRUCCIONES_AGENTE,
        tools=[registrar_movimiento, consultar_billetera, eliminar_movimiento],
    ),
)

mensaje = (
    "¡Me cayó la quincena de $650! Hoy gasté $4.50 en un Uber a la UTP, "
    "$7.25 en el almuerzo de la fonda y $3.99 en Spotify. "
    "Regístralo todo y dime cuánto saldo me queda."
)
print("Usuario:", mensaje)
respuesta = chat_billetera.send_message(mensaje)
print("\nSmart Wallet:")
print(respuesta.text)

In [ ]:
ver_tabla()

---
# Parte 5 · Ejercicio 3: Conexión a Yahoo Finance y Simulador

In [ ]:
import yfinance as yf

ALIAS_TICKERS = {
    "BITCOIN": "BTC-USD", "BTC": "BTC-USD",
    "ETHEREUM": "ETH-USD", "ETH": "ETH-USD",
    "NVIDIA": "NVDA", "APPLE": "AAPL", "MICROSOFT": "MSFT",
}

PRECIOS_RESPALDO = {
    "AAPL": 341.92, "NVDA": 218.52, "MSFT": 533.40, "SPY": 768.22,
    "BTC-USD": 81918.85, "ETH-USD": 2398.08,
}


def obtener_precio_activo(ticker: str) -> dict:
    """Consulta el precio más reciente, en dólares, de una acción, un ETF o una criptomoneda.

    Args:
        ticker: Símbolo de Yahoo Finance. Ejemplos: 'AAPL', 'NVDA', 'MSFT', 'SPY', 'BTC-USD', 'ETH-USD'.
    """
    simbolo = ticker.strip().upper()
    simbolo = ALIAS_TICKERS.get(simbolo, simbolo)
    print(f"   [YFINANCE] Consultando el precio de {simbolo}")

    try:
        historial = yf.Ticker(simbolo).history(period="5d")
        if historial.empty:
            raise ValueError("Sin datos")
        precio = round(float(historial["Close"].iloc[-1]), 2)
        fecha = historial.index[-1].strftime("%Y-%m-%d")
        return {"ok": True, "ticker": simbolo, "precio_usd": precio,
                "fecha_del_dato": fecha, "fuente": "Yahoo Finance"}
    except Exception:
        if simbolo in PRECIOS_RESPALDO:
            return {"ok": True, "ticker": simbolo, "precio_usd": PRECIOS_RESPALDO[simbolo],
                    "fuente": "RESPALDO OFFLINE (precio de referencia, no es en vivo)"}
        return {"ok": False, "error": f"No se pudo obtener el precio de {simbolo}."}


def simular_inversion(monto_inicial: float, aporte_mensual: float, tasa_anual_pct: float, anios: int) -> dict:
    """Calcula cuánto crecerá un ahorro o una inversión con interés compuesto mensual
    y aportes al final de cada mes.

    Args:
        monto_inicial: Capital inicial en dólares.
        aporte_mensual: Dinero que se agrega cada mes, en dólares (puede ser 0).
        tasa_anual_pct: Rendimiento anual esperado, en porcentaje. Ejemplo: 8.0 significa 8 %.
        anios: Cantidad de años de la inversión.
    """
    print(f"   [SIMULADOR] {anios} años al {tasa_anual_pct}% anual")

    # TODO 3.1: Calcula i (tasa mensual), n (meses totales) y valor_final con la fórmula de interés compuesto
    #           Pista: es la misma fórmula del Experimento 2. Si i es 0, valor_final = monto_inicial + aporte_mensual * n


    total_aportado = monto_inicial + aporte_mensual * n
    return {
        "ok": True,
        "valor_final_usd": round(valor_final, 2),
        "total_aportado_usd": round(total_aportado, 2),
        "ganancia_intereses_usd": round(valor_final - total_aportado, 2),
    }

In [ ]:
# TODO 3.2: Agrega las 5 herramientas creadas hasta ahora en la lista TODAS_LAS_HERRAMIENTAS
TODAS_LAS_HERRAMIENTAS = [

]

chat_completo = client.chats.create(
    model=MODELO,
    config=types.GenerateContentConfig(
        system_instruction=INSTRUCCIONES_AGENTE,
        tools=TODAS_LAS_HERRAMIENTAS,
    ),
)

pregunta_orquestada = (
    "Revisa cuánto saldo disponible tengo. Con el 20% de ese saldo quiero invertir: "
    "consulta cómo están NVIDIA y Apple, dime cuántas acciones de NVIDIA podría comprar con ese 20%, "
    "y cuánto tendría en 5 años si empiezo con ese mismo 20% y ahorro $50 al mes al 10% anual."
)
print("Usuario:", pregunta_orquestada)
respuesta = chat_completo.send_message(pregunta_orquestada)
print("\nSmart Wallet:")
print(respuesta.text)

---
# Parte 6 · Ejercicios 4 y 5: Salida estructurada (Pydantic) y Reto ITBMS

In [ ]:
from pydantic import BaseModel, Field


class GastoExtraido(BaseModel):
    # TODO 4.1: Define los campos descripcion (str), monto (float) y categoria (str) con Field(description=...)
    pass


class ListaGastos(BaseModel):
    comercio_principal: str = Field(description="Nombre del comercio principal, o 'varios'")
    gastos: list[GastoExtraido] = Field(description="Todos los gastos encontrados en el texto")


texto_desordenado = (
    "Ayer pasé por el Super 99 y compré leche a 2.45, pan de molde a 1.80 y un café de 6.99. "
    "Después pedí un Uber de 4.50 para la casa y en la noche pagué Netflix, 7.99."
)

respuesta_estructurada = client.models.generate_content(
    model=MODELO,
    contents=texto_desordenado,
    config=types.GenerateContentConfig(
        system_instruction="Extrae todos los gastos mencionados en el texto.",
        response_mime_type="application/json",
        response_schema=ListaGastos,
    ),
)

datos = respuesta_estructurada.parsed
if datos is None:
    datos = ListaGastos.model_validate_json(respuesta_estructurada.text)

print("Comercio:", datos.comercio_principal)
for gasto in datos.gastos:
    print(f"  - {gasto.descripcion}: ${gasto.monto:.2f} ({gasto.categoria})")

In [ ]:
for gasto in datos.gastos:
    registrar_movimiento("gasto", gasto.categoria, gasto.monto, gasto.descripcion)

ver_tabla()

In [ ]:
# EJERCICIO 5 (RETO INDIVIDUAL): Completa la herramienta calcular_itbms_panama
def calcular_itbms_panama(subtotal: float, porcentaje: int) -> dict:
    # TODO 5.1: Escribe el docstring (entre triples comillas): qué calcula la función y qué porcentaje usar.
    #           Pista: 7 para compras generales y 10 para licores u hospedaje.

    print(f"   [ITBMS] Calculando el {porcentaje}% sobre ${subtotal}")

    # TODO 5.2: Calcula el impuesto: subtotal por porcentaje, dividido entre 100. Redondea a 2 decimales.
    impuesto = 0.0

    # TODO 5.3: Calcula el total: subtotal más impuesto. Redondea a 2 decimales.
    total = 0.0

    return {"ok": True, "subtotal_usd": subtotal, "porcentaje": porcentaje,
            "itbms_usd": impuesto, "total_usd": total}

In [ ]:
prueba_1 = calcular_itbms_panama(45.00, 7)
prueba_2 = calcular_itbms_panama(3.00, 10)

if prueba_1["itbms_usd"] == 3.15 and prueba_1["total_usd"] == 48.15 and prueba_2["total_usd"] == 3.30:
    print("Correcto: tu herramienta calcula bien el ITBMS.")
else:
    print("Revisa tus fórmulas. Esperado: 45.00 al 7% = 3.15 de ITBMS y 48.15 en total; 3.00 al 10% = 3.30 en total.")

In [ ]:
TODAS_LAS_HERRAMIENTAS = [
    registrar_movimiento,
    consultar_billetera,
    eliminar_movimiento,
    obtener_precio_activo,
    simular_inversion,
    calcular_itbms_panama,
]

chat_itbms = client.chats.create(
    model=MODELO,
    config=types.GenerateContentConfig(
        system_instruction=INSTRUCCIONES_AGENTE,
        tools=TODAS_LAS_HERRAMIENTAS,
    ),
)

respuesta = chat_itbms.send_message("¿Cuánto pago en total por una compra de 45 dólares en el Super 99 con ITBMS?")
print(respuesta.text)

---
# Parte 7 · Ejercicio 6: Producción en Telegram
1. En Telegram, busca **@BotFather** y envía `/newbot`.
2. Asigna un nombre y un usuario terminado en `bot`.
3. Copia el token y guárdalo en Colab Secrets como `TELEGRAM_TOKEN` (activa **Notebook access**).

In [ ]:
import telebot

TELEGRAM_TOKEN = userdata.get("TELEGRAM_TOKEN")
bot = telebot.TeleBot(TELEGRAM_TOKEN, threaded=False)

yo = bot.get_me()
print(f"Bot conectado: @{yo.username}")
print(f"Ábrelo en tu celular: https://t.me/{yo.username}")

In [ ]:
import time
sesiones_usuarios = {}

def obtener_chat(user_id: int):
    if user_id not in sesiones_usuarios:
        sesiones_usuarios[user_id] = client.chats.create(
            model=MODELO,
            config=types.GenerateContentConfig(
                system_instruction=INSTRUCCIONES_AGENTE,
                tools=TODAS_LAS_HERRAMIENTAS,
            ),
        )
    return sesiones_usuarios[user_id]

@bot.message_handler(commands=["start", "ayuda"])
def comando_start(message):
    bienvenida = (
        f"Hola, {message.from_user.first_name}. Soy Smart Wallet, tu agente de finanzas.\n\n"
        "Escríbeme cosas como:\n"
        "- Me depositaron 300 de la beca del IFARHU\n"
        "- Gasté 2 en el Metro y 6.50 en el almuerzo de la fonda\n"
        "- ¿Cuánto saldo me queda y en qué gasto más?\n"
        "- ¿A cuánto están Bitcoin y NVIDIA hoy?\n"
        "- ¿Cuánto pago por una compra de 45 dólares con ITBMS?\n\n"
        "Comandos: /saldo /reset"
    )
    bot.reply_to(message, bienvenida)

@bot.message_handler(commands=["saldo"])
def comando_saldo(message):
    datos = consultar_billetera()
    texto = (
        f"Saldo disponible: ${datos['saldo_disponible_usd']:,.2f}\n"
        f"Ingresos: ${datos['ingresos_totales_usd']:,.2f}\n"
        f"Gastos: ${datos['gastos_totales_usd']:,.2f}"
    )
    bot.reply_to(message, texto)

@bot.message_handler(commands=["reset"])
def comando_reset(message):
    sesiones_usuarios.pop(message.from_user.id, None)
    bot.reply_to(message, "Conversación reiniciada. Tus movimientos siguen guardados en la base de datos.")

@bot.message_handler(content_types=["text"])
def manejar_mensaje(message):
    print(f"\n[TELEGRAM] {message.from_user.first_name}: {message.text}")
    bot.send_chat_action(message.chat.id, "typing")
    try:
        # TODO 6.1: Obtén el chat de este usuario con obtener_chat(message.from_user.id)
        #           y envíale el texto con chat.send_message(message.text). Guarda el resultado en respuesta.

        texto = respuesta.text
        if not texto:
            texto = "No tengo una respuesta para eso. Intenta preguntarlo de otra forma."
        bot.reply_to(message, texto)
    except Exception as error:
        print("[ERROR]", error)
        bot.reply_to(message, "Ocurrió un error procesando tu mensaje. Verifica haber completado el TODO 6.1.")

print(f"@{yo.username} está en línea. Escríbele desde tu celular.")
print("Para apagar el bot, presiona el botón de detener de esta celda.\n")

# Evitamos bloqueos de webhooks previos y forzamos reconexión en bucle
bot.remove_webhook()

while True:
    try:
        bot.polling(none_stop=True, interval=0, timeout=20)
    except Exception as e:
        print(f"[RECONECTANDO] Ocurrió un error: {e}. Reintentando en 5 segundos...")
        time.sleep(5)